In [3]:
import sys
from importlib.util import find_spec

print("Python do notebook:", sys.executable)

for modulo in ["spacy", "pt_core_news_sm"]:
    disponivel = find_spec(modulo) is not None
    print(f"{modulo}: {'disponível' if disponivel else 'não encontrado'}")

Python do notebook: /opt/homebrew/anaconda3/bin/python
spacy: disponível
pt_core_news_sm: disponível


In [4]:
import spacy

nlp = spacy.load("pt_core_news_sm")

print("Modelo carregado:", nlp.meta["name"])
print("Versão do spaCy:", spacy.__version__)
print("Versão do modelo:", nlp.meta["version"])
print("Categorias do NER:", nlp.get_pipe("ner").labels)

Modelo carregado: core_news_sm
Versão do spaCy: 3.8.16
Versão do modelo: 3.8.0
Categorias do NER: ('LOC', 'MISC', 'ORG', 'PER')


In [5]:
texto = "Como solicitar o trancamento de matrícula no Ibmec Faria Lima?"

doc = nlp(texto)

print("Texto:", texto)
print("\nEntidades encontradas:")

if doc.ents:
    for entidade in doc.ents:
        print(
            entidade.text,
            "->", entidade.label_,
            "| posição:", entidade.start_char, entidade.end_char,
        )
else:
    print("Nenhuma entidade encontrada.")

Texto: Como solicitar o trancamento de matrícula no Ibmec Faria Lima?

Entidades encontradas:
Ibmec Faria Lima -> LOC | posição: 45 61


In [6]:
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

In [7]:
# Pipeline em português, inicialmente sem modelo treinado.
nlp_regras = spacy.blank("pt")

ruler = nlp_regras.add_pipe(
    "entity_ruler",
    config={"phrase_matcher_attr": "LOWER"},
)

ruler.add_patterns([
    {
        "label": "SERVICO_ACADEMICO",
        "pattern": "matrícula",
    }
])

doc_regras = nlp_regras(texto)

for entidade in doc_regras.ents:
    print(entidade.text, "->", entidade.label_)

matrícula -> SERVICO_ACADEMICO


In [8]:
ruler.add_patterns([
    {"label": "INSTITUICAO", "pattern": "Ibmec"},
    {"label": "UNIDADE", "pattern": "Faria Lima"},
])

for nome, pipeline in [
    ("MODELO PRÉ-TREINADO", nlp),
    ("REGRAS DO PROJETO", nlp_regras),
]:
    print(f"\n{nome}")

    documento = pipeline(texto)

    for entidade in documento.ents:
        print(entidade.text, "->", entidade.label_)


MODELO PRÉ-TREINADO
Ibmec Faria Lima -> LOC

REGRAS DO PROJETO
matrícula -> SERVICO_ACADEMICO
Ibmec -> INSTITUICAO
Faria Lima -> UNIDADE


In [9]:
perguntas_exploratorias = [
    "Como cancelar <minha matrícula?",
    "Como trancar minha MATRÍCULA no IBMEC FARIA LIMA?",
    "Como reabrir minha matricula?",
    "Onde consulto as matrículas anteriores?",
    "Como conseguir um estágio no Ibmec?",
    "O prazo de matrícula já terminou?",
]

for pergunta in perguntas_exploratorias:
    documento = nlp_regras(pergunta)

    entidades = [
        (entidade.text, entidade.label_)
        for entidade in documento.ents
    ]

    print("\nPergunta:", pergunta)
    print("Entidades:", entidades)


Pergunta: Como cancelar <minha matrícula?
Entidades: [('matrícula', 'SERVICO_ACADEMICO')]

Pergunta: Como trancar minha MATRÍCULA no IBMEC FARIA LIMA?
Entidades: [('MATRÍCULA', 'SERVICO_ACADEMICO'), ('IBMEC', 'INSTITUICAO'), ('FARIA LIMA', 'UNIDADE')]

Pergunta: Como reabrir minha matricula?
Entidades: []

Pergunta: Onde consulto as matrículas anteriores?
Entidades: []

Pergunta: Como conseguir um estágio no Ibmec?
Entidades: [('Ibmec', 'INSTITUICAO')]

Pergunta: O prazo de matrícula já terminou?
Entidades: [('matrícula', 'SERVICO_ACADEMICO')]


In [10]:
variantes_matricula = [
    "matricula",
    "matrículas",
    "matriculas",
]

ruler.add_patterns([
    {
        "label": "SERVICO_ACADEMICO",
        "pattern": variante,
    }
    for variante in variantes_matricula
])

for pergunta in perguntas_exploratorias:
    documento = nlp_regras(pergunta)

    print("\nPergunta:", pergunta)
    print("Entidades:", [
        (entidade.text, entidade.label_)
        for entidade in documento.ents
    ])


Pergunta: Como cancelar <minha matrícula?
Entidades: [('matrícula', 'SERVICO_ACADEMICO')]

Pergunta: Como trancar minha MATRÍCULA no IBMEC FARIA LIMA?
Entidades: [('MATRÍCULA', 'SERVICO_ACADEMICO'), ('IBMEC', 'INSTITUICAO'), ('FARIA LIMA', 'UNIDADE')]

Pergunta: Como reabrir minha matricula?
Entidades: [('matricula', 'SERVICO_ACADEMICO')]

Pergunta: Onde consulto as matrículas anteriores?
Entidades: [('matrículas', 'SERVICO_ACADEMICO')]

Pergunta: Como conseguir um estágio no Ibmec?
Entidades: [('Ibmec', 'INSTITUICAO')]

Pergunta: O prazo de matrícula já terminou?
Entidades: [('matrícula', 'SERVICO_ACADEMICO')]


In [11]:
termos_academicos = [
    "estágio",
    "estagio",
    "estágios",
    "estagios",
    "carteirinha",
    "carteirinhas",
]

ruler.add_patterns([
    {"label": "SERVICO_ACADEMICO", "pattern": termo}
    for termo in termos_academicos
])

novas_perguntas = [
    "Como conseguir um estágio no Ibmec?",
    "Onde envio os documentos de estagio?",
    "Como solicitar minha carteirinha na Faria Lima?",
    "Posso usar carteirinhas digitais?",
]

for pergunta in novas_perguntas:
    documento = nlp_regras(pergunta)

    print("\nPergunta:", pergunta)
    print("Entidades:", [
        (entidade.text, entidade.label_)
        for entidade in documento.ents
    ])


Pergunta: Como conseguir um estágio no Ibmec?
Entidades: [('estágio', 'SERVICO_ACADEMICO'), ('Ibmec', 'INSTITUICAO')]

Pergunta: Onde envio os documentos de estagio?
Entidades: [('estagio', 'SERVICO_ACADEMICO')]

Pergunta: Como solicitar minha carteirinha na Faria Lima?
Entidades: [('carteirinha', 'SERVICO_ACADEMICO'), ('Faria Lima', 'UNIDADE')]

Pergunta: Posso usar carteirinhas digitais?
Entidades: [('carteirinhas', 'SERVICO_ACADEMICO')]


In [12]:
SERVICO = "SERVICO_ACADEMICO"

casos_avaliacao = [
    (
        "Preciso reabrir a matrícula no Ibmec.",
        [("matrícula", SERVICO), ("Ibmec", "INSTITUICAO")],
    ),
    (
        "Na Faria Lima, como peço minha carteirinha?",
        [("Faria Lima", "UNIDADE"), ("carteirinha", SERVICO)],
    ),
    (
        "Quais documentos são exigidos para estágios?",
        [("estágios", SERVICO)],
    ),
    (
        "Há orientações sobre matriculas no IBMEC?",
        [("matriculas", SERVICO), ("IBMEC", "INSTITUICAO")],
    ),
    (
        "Preciso renovar a matrícula e a carteirinha.",
        [("matrícula", SERVICO), ("carteirinha", SERVICO)],
    ),
    ("O sistema matricula os alunos automaticamente.", []),
    ("Eu estagio em uma empresa parceira.", []),
    ("Como posso obter ajuda?", []),
]

for numero, (frase, esperado) in enumerate(casos_avaliacao, start=1):
    print(f"\n{numero}. {frase}")
    print("Esperado:", esperado)


1. Preciso reabrir a matrícula no Ibmec.
Esperado: [('matrícula', 'SERVICO_ACADEMICO'), ('Ibmec', 'INSTITUICAO')]

2. Na Faria Lima, como peço minha carteirinha?
Esperado: [('Faria Lima', 'UNIDADE'), ('carteirinha', 'SERVICO_ACADEMICO')]

3. Quais documentos são exigidos para estágios?
Esperado: [('estágios', 'SERVICO_ACADEMICO')]

4. Há orientações sobre matriculas no IBMEC?
Esperado: [('matriculas', 'SERVICO_ACADEMICO'), ('IBMEC', 'INSTITUICAO')]

5. Preciso renovar a matrícula e a carteirinha.
Esperado: [('matrícula', 'SERVICO_ACADEMICO'), ('carteirinha', 'SERVICO_ACADEMICO')]

6. O sistema matricula os alunos automaticamente.
Esperado: []

7. Eu estagio em uma empresa parceira.
Esperado: []

8. Como posso obter ajuda?
Esperado: []


In [13]:
from collections import Counter

totais_ner = Counter()
resultados_avaliacao = []

for numero, (frase, anotacoes) in enumerate(casos_avaliacao, start=1):
    esperados = set()

    for trecho, rotulo in anotacoes:
        # Estes exemplos têm uma única ocorrência de cada trecho.
        assert frase.count(trecho) == 1

        inicio = frase.index(trecho)
        esperados.add((inicio, inicio + len(trecho), rotulo))

    documento = nlp_regras(frase)

    previstos = {
        (entidade.start_char, entidade.end_char, entidade.label_)
        for entidade in documento.ents
    }

    acertos = previstos & esperados
    falsos_positivos = previstos - esperados
    falsos_negativos = esperados - previstos

    contagens = {
        "VP": len(acertos),
        "FP": len(falsos_positivos),
        "FN": len(falsos_negativos),
    }
    totais_ner.update(contagens)

    resultados_avaliacao.append({
        "texto": frase,
        "esperados": esperados,
        "previstos": previstos,
        **contagens,
    })

    print(f"\n{numero}. {frase}")
    print(contagens)

    for inicio, fim, rotulo in sorted(falsos_positivos):
        print("Reconheceu indevidamente:", frase[inicio:fim], "->", rotulo)

    for inicio, fim, rotulo in sorted(falsos_negativos):
        print("Deixou de reconhecer:", frase[inicio:fim], "->", rotulo)

print("\nTOTAIS:", dict(totais_ner))


1. Preciso reabrir a matrícula no Ibmec.
{'VP': 2, 'FP': 0, 'FN': 0}

2. Na Faria Lima, como peço minha carteirinha?
{'VP': 2, 'FP': 0, 'FN': 0}

3. Quais documentos são exigidos para estágios?
{'VP': 1, 'FP': 0, 'FN': 0}

4. Há orientações sobre matriculas no IBMEC?
{'VP': 2, 'FP': 0, 'FN': 0}

5. Preciso renovar a matrícula e a carteirinha.
{'VP': 2, 'FP': 0, 'FN': 0}

6. O sistema matricula os alunos automaticamente.
{'VP': 0, 'FP': 1, 'FN': 0}
Reconheceu indevidamente: matricula -> SERVICO_ACADEMICO

7. Eu estagio em uma empresa parceira.
{'VP': 0, 'FP': 1, 'FN': 0}
Reconheceu indevidamente: estagio -> SERVICO_ACADEMICO

8. Como posso obter ajuda?
{'VP': 0, 'FP': 0, 'FN': 0}

TOTAIS: {'VP': 9, 'FP': 2, 'FN': 0}


In [14]:
vp = totais_ner["VP"]
fp = totais_ner["FP"]
fn = totais_ner["FN"]

precisao = vp / (vp + fp) if vp + fp else 0.0
revocacao = vp / (vp + fn) if vp + fn else 0.0

f1 = (
    2 * precisao * revocacao / (precisao + revocacao)
    if precisao + revocacao else 0.0
)

print(f"Precisão: {precisao:.2%}")
print(f"Revocação: {revocacao:.2%}")
print(f"F1: {f1:.2%}")

Precisão: 81.82%
Revocação: 100.00%
F1: 90.00%


In [15]:
rotulos = ["INSTITUICAO", "UNIDADE", "SERVICO_ACADEMICO"]
metricas_por_rotulo = []

for rotulo in rotulos:
    vp = fp = fn = 0

    for resultado in resultados_avaliacao:
        esperados = {
            entidade for entidade in resultado["esperados"]
            if entidade[2] == rotulo
        }
        previstos = {
            entidade for entidade in resultado["previstos"]
            if entidade[2] == rotulo
        }

        vp += len(previstos & esperados)
        fp += len(previstos - esperados)
        fn += len(esperados - previstos)

    precisao_rotulo = vp / (vp + fp) if vp + fp else 0.0
    revocacao_rotulo = vp / (vp + fn) if vp + fn else 0.0
    f1_rotulo = 2 * vp / (2 * vp + fp + fn) if 2 * vp + fp + fn else 0.0

    metricas_por_rotulo.append({
        "rotulo": rotulo,
        "VP": vp,
        "FP": fp,
        "FN": fn,
        "suporte": vp + fn,
        "precisao": precisao_rotulo,
        "revocacao": revocacao_rotulo,
        "f1": f1_rotulo,
    })

    print(
        f"{rotulo} | suporte={vp + fn} | VP={vp} FP={fp} FN={fn} | "
        f"P={precisao_rotulo:.2%} R={revocacao_rotulo:.2%} "
        f"F1={f1_rotulo:.2%}"
    )

INSTITUICAO | suporte=2 | VP=2 FP=0 FN=0 | P=100.00% R=100.00% F1=100.00%
UNIDADE | suporte=1 | VP=1 FP=0 FN=0 | P=100.00% R=100.00% F1=100.00%
SERVICO_ACADEMICO | suporte=6 | VP=6 FP=2 FN=0 | P=75.00% R=100.00% F1=85.71%


In [16]:
padroes_congelados = [
    {"label": rotulo, "pattern": padrao}
    for rotulo, padrao in sorted({
        (regra["label"], regra["pattern"])
        for regra in ruler.patterns
    })
]

configuracao_experimento = {
    "versao_experimento": "ner_basico_v1",
    "abordagem_avaliada": "Regras com EntityRuler",
    "idioma": "pt",
    "comparacao_texto": "LOWER",
    "versao_spacy": spacy.__version__,
    "modelo_pre_treinado_comparado": nlp.meta["name"],
    "versao_modelo_pre_treinado": nlp.meta["version"],
    "quantidade_casos": len(casos_avaliacao),
    "tipo_avaliacao": "Piloto com exemplos sintéticos",
    "origem_anotacoes": "Propostas com apoio de IA",
}

print("Regras distintas:", len(padroes_congelados))
print("Casos avaliados:", configuracao_experimento["quantidade_casos"])
print("Abordagem avaliada:", configuracao_experimento["abordagem_avaliada"])

Regras distintas: 12
Casos avaliados: 8
Abordagem avaliada: Regras com EntityRuler


In [17]:
from pathlib import Path
import csv

pasta_atual = Path.cwd()
raiz = pasta_atual.parent if pasta_atual.name == "notebooks" else pasta_atual
pasta_resultados = raiz / "reports" / "ner_basico_v1"

linhas_casos = []
linhas_entidades = []

for numero, resultado in enumerate(resultados_avaliacao, start=1):
    linhas_casos.append({
        "caso": numero,
        "texto": resultado["texto"],
        "VP": resultado["VP"],
        "FP": resultado["FP"],
        "FN": resultado["FN"],
    })

    for origem, campo in [
        ("referencia", "esperados"),
        ("previsao", "previstos"),
    ]:
        for inicio, fim, rotulo in sorted(resultado[campo]):
            linhas_entidades.append({
                "caso": numero,
                "origem": origem,
                "inicio": inicio,
                "fim": fim,
                "texto_entidade": resultado["texto"][inicio:fim],
                "rotulo": rotulo,
            })

vp, fp, fn = (
    totais_ner[chave] for chave in ["VP", "FP", "FN"]
)

metrica_global = {
    "rotulo": "GLOBAL_MICRO",
    "VP": vp,
    "FP": fp,
    "FN": fn,
    "suporte": vp + fn,
    "precisao": vp / (vp + fp) if vp + fp else 0.0,
    "revocacao": vp / (vp + fn) if vp + fn else 0.0,
    "f1": 2 * vp / (2 * vp + fp + fn) if 2 * vp + fp + fn else 0.0,
}

configuracao_salva = configuracao_experimento.copy()
configuracao_salva["revisao_humana_anotacoes"] = "Não registrada"
configuracao_salva["quantidade_padroes"] = len(padroes_congelados)

saidas_ner = {
    "padroes.csv": padroes_congelados,
    "casos.csv": linhas_casos,
    "entidades.csv": linhas_entidades,
    "metricas.csv": [metrica_global] + metricas_por_rotulo,
    "configuracao.csv": [configuracao_salva],
}

pasta_resultados.mkdir(parents=True, exist_ok=True)

for nome in saidas_ner:
    if (pasta_resultados / nome).exists():
        raise FileExistsError(f"{nome} já existe; nada foi sobrescrito.")

for nome, linhas in saidas_ner.items():
    colunas = list(dict.fromkeys(
        campo for linha in linhas for campo in linha
    ))
    esperadas = [
        {campo: str(linha.get(campo, "")) for campo in colunas}
        for linha in linhas
    ]

    caminho = pasta_resultados / nome

    with caminho.open("x", encoding="utf-8-sig", newline="") as arquivo:
        escritor = csv.DictWriter(
            arquivo, fieldnames=colunas, delimiter=";"
        )
        escritor.writeheader()
        escritor.writerows(esperadas)

    with caminho.open("r", encoding="utf-8-sig", newline="") as arquivo:
        recuperadas = list(csv.DictReader(arquivo, delimiter=";"))

    assert recuperadas == esperadas, f"Falha na gravação de {nome}"
    print(f"{nome}: {len(recuperadas)} registros — conferido")

FileExistsError: padroes.csv já existe; nada foi sobrescrito.

In [18]:
def ler_resultado_ner(nome):
    with (pasta_resultados / nome).open(
        "r", encoding="utf-8-sig", newline=""
    ) as arquivo:
        return list(csv.DictReader(arquivo, delimiter=";"))


configuracao_lida = ler_resultado_ner("configuracao.csv")[0]
padroes_lidos = ler_resultado_ner("padroes.csv")
casos_lidos = ler_resultado_ner("casos.csv")
entidades_lidas = ler_resultado_ner("entidades.csv")

nlp_recarregado = spacy.blank(configuracao_lida["idioma"])

ruler_recarregado = nlp_recarregado.add_pipe(
    "entity_ruler",
    config={
        "phrase_matcher_attr": configuracao_lida["comparacao_texto"]
    },
)
ruler_recarregado.add_patterns(padroes_lidos)

for caso in casos_lidos:
    previsoes_salvas = {
        (int(entidade["inicio"]), int(entidade["fim"]), entidade["rotulo"])
        for entidade in entidades_lidas
        if entidade["caso"] == caso["caso"]
        and entidade["origem"] == "previsao"
    }

    documento = nlp_recarregado(caso["texto"])

    previsoes_reproduzidas = {
        (entidade.start_char, entidade.end_char, entidade.label_)
        for entidade in documento.ents
    }

    assert previsoes_reproduzidas == previsoes_salvas, (
        f"Previsões diferentes no caso {caso['caso']}"
    )

print("Reprodução conferida nos 8 casos.")

Reprodução conferida nos 8 casos.


In [19]:
from pathlib import Path
import csv

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

pasta = raiz / "reports" / "ner_basico_v1"

def ler_csv(nome):
    with (pasta / nome).open(
        encoding="utf-8-sig", newline=""
    ) as arquivo:
        return list(csv.DictReader(arquivo, delimiter=";"))

casos = ler_csv("casos.csv")
entidades = ler_csv("entidades.csv")

for caso in casos:
    esperadas = [
        (entidade["texto_entidade"], entidade["rotulo"])
        for entidade in entidades
        if entidade["caso"] == caso["caso"]
        and entidade["origem"] == "referencia"
    ]

    print(f'{caso["caso"]}. {caso["texto"]}')
    print("Entidades esperadas:", esperadas)
    print()

1. Preciso reabrir a matrícula no Ibmec.
Entidades esperadas: [('matrícula', 'SERVICO_ACADEMICO'), ('Ibmec', 'INSTITUICAO')]

2. Na Faria Lima, como peço minha carteirinha?
Entidades esperadas: [('Faria Lima', 'UNIDADE'), ('carteirinha', 'SERVICO_ACADEMICO')]

3. Quais documentos são exigidos para estágios?
Entidades esperadas: [('estágios', 'SERVICO_ACADEMICO')]

4. Há orientações sobre matriculas no IBMEC?
Entidades esperadas: [('matriculas', 'SERVICO_ACADEMICO'), ('IBMEC', 'INSTITUICAO')]

5. Preciso renovar a matrícula e a carteirinha.
Entidades esperadas: [('matrícula', 'SERVICO_ACADEMICO'), ('carteirinha', 'SERVICO_ACADEMICO')]

6. O sistema matricula os alunos automaticamente.
Entidades esperadas: []

7. Eu estagio em uma empresa parceira.
Entidades esperadas: []

8. Como posso obter ajuda?
Entidades esperadas: []



In [20]:
conferidas = 0

for entidade in entidades:
    if entidade["origem"] != "referencia":
        continue

    caso = next(
        caso for caso in casos
        if caso["caso"] == entidade["caso"]
    )

    inicio = int(entidade["inicio"])
    fim = int(entidade["fim"])
    trecho = caso["texto"][inicio:fim]

    assert trecho == entidade["texto_entidade"], (
        f'Posição incorreta no caso {caso["caso"]}: {trecho!r}'
    )

    conferidas += 1

print(f"Entidades de referência com posições conferidas: {conferidas}")

Entidades de referência com posições conferidas: 9


In [21]:
casos_contexto = [
    {
        "texto": "A faculdade matricula os estudantes pelo sistema.",
        "esperado": [],
    },
    {
        "texto": "Preciso do comprovante de matricula.",
        "esperado": [("matricula", "SERVICO_ACADEMICO")],
    },
    {
        "texto": "Atualmente eu estagio em um banco.",
        "esperado": [],
    },
    {
        "texto": "Onde encontro o contrato de estagio?",
        "esperado": [("estagio", "SERVICO_ACADEMICO")],
    },
]

for caso in casos_contexto:
    documento = nlp_regras(caso["texto"])
    previstas = [
        (entidade.text, entidade.label_)
        for entidade in documento.ents
    ]

    print(caso["texto"])
    print("Esperado:", caso["esperado"])
    print("Previsto:", previstas)
    print()

A faculdade matricula os estudantes pelo sistema.
Esperado: []
Previsto: [('matricula', 'SERVICO_ACADEMICO')]

Preciso do comprovante de matricula.
Esperado: [('matricula', 'SERVICO_ACADEMICO')]
Previsto: [('matricula', 'SERVICO_ACADEMICO')]

Atualmente eu estagio em um banco.
Esperado: []
Previsto: [('estagio', 'SERVICO_ACADEMICO')]

Onde encontro o contrato de estagio?
Esperado: [('estagio', 'SERVICO_ACADEMICO')]
Previsto: [('estagio', 'SERVICO_ACADEMICO')]



In [22]:
for caso in casos_contexto:
    documento = nlp(caso["texto"])

    print(caso["texto"])

    for token in documento:
        if token.lower_ in {"matricula", "estagio"}:
            print(
                f"Palavra: {token.text} | "
                f"Classe: {token.pos_} | "
                f"Lema: {token.lemma_}"
            )

    print()

A faculdade matricula os estudantes pelo sistema.
Palavra: matricula | Classe: VERB | Lema: matricula

Preciso do comprovante de matricula.
Palavra: matricula | Classe: NOUN | Lema: matricula

Atualmente eu estagio em um banco.
Palavra: estagio | Classe: VERB | Lema: estagio

Onde encontro o contrato de estagio?
Palavra: estagio | Classe: NOUN | Lema: estagio



In [23]:
def reconhecer_entidades_v2(texto):
    documento_regras = nlp_regras(texto)
    documento_gramatical = nlp(texto)

    entidades_aceitas = []

    for entidade in documento_regras.ents:
        ambiguo = (
            entidade.label_ == "SERVICO_ACADEMICO"
            and entidade.text.lower() in {"matricula", "estagio"}
        )

        if ambiguo:
            trecho = documento_gramatical.char_span(
                entidade.start_char,
                entidade.end_char,
                alignment_mode="strict",
            )

            if trecho is not None:
                identificado_como_verbo = any(
                    token.pos_ == "VERB" for token in trecho
                )

                precedido_por_eu = (
                    trecho.start > 0
                    and documento_gramatical[
                        trecho.start - 1
                    ].lower_ == "eu"
                )

                if identificado_como_verbo or precedido_por_eu:
                    continue

        entidades_aceitas.append(
            (entidade.text, entidade.label_)
        )

    return entidades_aceitas

In [24]:
casos_regressao = ler_csv("casos.csv")
entidades_regressao = ler_csv("entidades.csv")

for caso in casos_regressao:
    esperadas = [
        (entidade["texto_entidade"], entidade["rotulo"])
        for entidade in entidades_regressao
        if entidade["caso"] == caso["caso"]
        and entidade["origem"] == "referencia"
    ]

    previstas = reconhecer_entidades_v2(caso["texto"])

    print(f'{caso["caso"]}. {caso["texto"]}')
    print("Esperado:", esperadas)
    print("Previsto v2:", previstas)
    print(
        "Conferência:",
        "OK" if previstas == esperadas else "REVISAR",
    )
    print()

1. Preciso reabrir a matrícula no Ibmec.
Esperado: [('matrícula', 'SERVICO_ACADEMICO'), ('Ibmec', 'INSTITUICAO')]
Previsto v2: [('matrícula', 'SERVICO_ACADEMICO'), ('Ibmec', 'INSTITUICAO')]
Conferência: OK

2. Na Faria Lima, como peço minha carteirinha?
Esperado: [('Faria Lima', 'UNIDADE'), ('carteirinha', 'SERVICO_ACADEMICO')]
Previsto v2: [('Faria Lima', 'UNIDADE'), ('carteirinha', 'SERVICO_ACADEMICO')]
Conferência: OK

3. Quais documentos são exigidos para estágios?
Esperado: [('estágios', 'SERVICO_ACADEMICO')]
Previsto v2: [('estágios', 'SERVICO_ACADEMICO')]
Conferência: OK

4. Há orientações sobre matriculas no IBMEC?
Esperado: [('matriculas', 'SERVICO_ACADEMICO'), ('IBMEC', 'INSTITUICAO')]
Previsto v2: [('matriculas', 'SERVICO_ACADEMICO'), ('IBMEC', 'INSTITUICAO')]
Conferência: OK

5. Preciso renovar a matrícula e a carteirinha.
Esperado: [('matrícula', 'SERVICO_ACADEMICO'), ('carteirinha', 'SERVICO_ACADEMICO')]
Previsto v2: [('matrícula', 'SERVICO_ACADEMICO'), ('carteirinha', 'S

In [25]:
divergencias = []

for caso in casos_regressao:
    esperadas = [
        (entidade["texto_entidade"], entidade["rotulo"])
        for entidade in entidades_regressao
        if entidade["caso"] == caso["caso"]
        and entidade["origem"] == "referencia"
    ]

    previstas = reconhecer_entidades_v2(caso["texto"])

    if previstas != esperadas:
        divergencias.append(caso["caso"])

print("Casos conferidos:", len(casos_regressao))
print("Casos sem divergência:", len(casos_regressao) - len(divergencias))
print("Casos para revisar:", divergencias)

Casos conferidos: 8
Casos sem divergência: 8
Casos para revisar: []


In [26]:
caso_7 = next(
    caso for caso in casos_regressao
    if caso["caso"] == "7"
)

texto_caso_7 = caso_7["texto"]

print("Texto:", texto_caso_7)
print("Previsto v2:", reconhecer_entidades_v2(texto_caso_7))

for token in nlp(texto_caso_7):
    print(f"{token.text:<15} classe={token.pos_}")

Texto: Eu estagio em uma empresa parceira.
Previsto v2: []
Eu              classe=PRON
estagio         classe=NOUN
em              classe=ADP
uma             classe=DET
empresa         classe=NOUN
parceira        classe=ADJ
.               classe=PUNCT


In [27]:
sondagem_contexto = [
    "Eu estagio em uma empresa parceira.",
    "Eu estagio numa biblioteca.",
    "Você estagia em qual empresa?",
    "O estágio acontece em uma empresa parceira.",
    "Meu estagio começa amanhã.",
    "Como solicitar um estagio?",
]

for texto_sondagem in sondagem_contexto:
    documento = nlp(texto_sondagem)

    analise = [
        (token.text, token.pos_)
        for token in documento
        if token.lower_.startswith("estagi")
        or token.lower_.startswith("estági")
    ]

    print(texto_sondagem)
    print("Análise gramatical:", analise)
    print("Entidades v2:", reconhecer_entidades_v2(texto_sondagem))
    print()

Eu estagio em uma empresa parceira.
Análise gramatical: [('estagio', 'NOUN')]
Entidades v2: []

Eu estagio numa biblioteca.
Análise gramatical: [('estagio', 'VERB')]
Entidades v2: []

Você estagia em qual empresa?
Análise gramatical: [('estagia', 'VERB')]
Entidades v2: []

O estágio acontece em uma empresa parceira.
Análise gramatical: [('estágio', 'NOUN')]
Entidades v2: [('estágio', 'SERVICO_ACADEMICO')]

Meu estagio começa amanhã.
Análise gramatical: [('estagio', 'NOUN')]
Entidades v2: [('estagio', 'SERVICO_ACADEMICO')]

Como solicitar um estagio?
Análise gramatical: [('estagio', 'NOUN')]
Entidades v2: [('estagio', 'SERVICO_ACADEMICO')]



### NER v2: regras com análise de contexto

Combinamos as regras do EntityRuler com a análise gramatical
do modelo pt_core_news_sm.

Para as formas ambíguas "matricula" e "estagio", descartamos
a entidade quando a palavra é classificada como verbo ou
quando aparece imediatamente depois de "eu".

A versão 2 apresentou correspondência com as entidades esperadas
nos oito casos iniciais. Esses casos foram usados no desenvolvimento,
portanto esse resultado não representa uma avaliação independente.

A análise gramatical pode errar, e a regra com "eu" cobre apenas
um contexto específico.

In [28]:
casos_teste_v2 = [
    {
        "texto": "Onde solicito minha carteirinha no Ibmec?",
        "esperado": [
            ("carteirinha", "SERVICO_ACADEMICO"),
            ("Ibmec", "INSTITUICAO"),
        ],
    },
    {
        "texto": "Há vagas de estágio na Faria Lima?",
        "esperado": [
            ("estágio", "SERVICO_ACADEMICO"),
            ("Faria Lima", "UNIDADE"),
        ],
    },
    {
        "texto": "Quero informações sobre matriculas e estágios.",
        "esperado": [
            ("matriculas", "SERVICO_ACADEMICO"),
            ("estágios", "SERVICO_ACADEMICO"),
        ],
    },
    {
        "texto": "Consigo renovar a matrícula pelo celular?",
        "esperado": [
            ("matrícula", "SERVICO_ACADEMICO"),
        ],
    },
    {
        "texto": "O atendente matricula novos alunos diariamente.",
        "esperado": [],
    },
    {
        "texto": "Eu também estagio em uma empresa.",
        "esperado": [],
    },
    {
        "texto": "Onde fica a secretaria?",
        "esperado": [],
    },
    {
        "texto": "As carteirinhas do IBMEC ficam prontas quando?",
        "esperado": [
            ("carteirinhas", "SERVICO_ACADEMICO"),
            ("IBMEC", "INSTITUICAO"),
        ],
    },
]

for numero, caso_teste in enumerate(casos_teste_v2, start=1):
    print(f'{numero}. {caso_teste["texto"]}')
    print("Esperado:", caso_teste["esperado"])
    print()

1. Onde solicito minha carteirinha no Ibmec?
Esperado: [('carteirinha', 'SERVICO_ACADEMICO'), ('Ibmec', 'INSTITUICAO')]

2. Há vagas de estágio na Faria Lima?
Esperado: [('estágio', 'SERVICO_ACADEMICO'), ('Faria Lima', 'UNIDADE')]

3. Quero informações sobre matriculas e estágios.
Esperado: [('matriculas', 'SERVICO_ACADEMICO'), ('estágios', 'SERVICO_ACADEMICO')]

4. Consigo renovar a matrícula pelo celular?
Esperado: [('matrícula', 'SERVICO_ACADEMICO')]

5. O atendente matricula novos alunos diariamente.
Esperado: []

6. Eu também estagio em uma empresa.
Esperado: []

7. Onde fica a secretaria?
Esperado: []

8. As carteirinhas do IBMEC ficam prontas quando?
Esperado: [('carteirinhas', 'SERVICO_ACADEMICO'), ('IBMEC', 'INSTITUICAO')]



In [29]:
resultados_teste_v2 = []

for numero, caso_teste in enumerate(casos_teste_v2, start=1):
    previstas = reconhecer_entidades_v2(caso_teste["texto"])

    resultados_teste_v2.append({
        "caso": numero,
        "texto": caso_teste["texto"],
        "esperadas": caso_teste["esperado"],
        "previstas": previstas,
    })

    print(f'{numero}. {caso_teste["texto"]}')
    print("Esperado:", caso_teste["esperado"])
    print("Previsto:", previstas)
    print(
        "Conferência:",
        "OK" if previstas == caso_teste["esperado"] else "REVISAR",
    )
    print()

1. Onde solicito minha carteirinha no Ibmec?
Esperado: [('carteirinha', 'SERVICO_ACADEMICO'), ('Ibmec', 'INSTITUICAO')]
Previsto: [('carteirinha', 'SERVICO_ACADEMICO'), ('Ibmec', 'INSTITUICAO')]
Conferência: OK

2. Há vagas de estágio na Faria Lima?
Esperado: [('estágio', 'SERVICO_ACADEMICO'), ('Faria Lima', 'UNIDADE')]
Previsto: [('estágio', 'SERVICO_ACADEMICO'), ('Faria Lima', 'UNIDADE')]
Conferência: OK

3. Quero informações sobre matriculas e estágios.
Esperado: [('matriculas', 'SERVICO_ACADEMICO'), ('estágios', 'SERVICO_ACADEMICO')]
Previsto: [('matriculas', 'SERVICO_ACADEMICO'), ('estágios', 'SERVICO_ACADEMICO')]
Conferência: OK

4. Consigo renovar a matrícula pelo celular?
Esperado: [('matrícula', 'SERVICO_ACADEMICO')]
Previsto: [('matrícula', 'SERVICO_ACADEMICO')]
Conferência: OK

5. O atendente matricula novos alunos diariamente.
Esperado: []
Previsto: []
Conferência: OK

6. Eu também estagio em uma empresa.
Esperado: []
Previsto: [('estagio', 'SERVICO_ACADEMICO')]
Conferência

In [30]:
def converter_em_posicoes(texto, anotacoes):
    posicoes = set()

    for trecho, rotulo in anotacoes:
        assert texto.count(trecho) == 1, (
            f"Trecho ausente ou repetido: {trecho!r}"
        )

        inicio = texto.index(trecho)
        posicoes.add((inicio, inicio + len(trecho), rotulo))

    return posicoes


vp_v2 = fp_v2 = fn_v2 = 0
casos_corretos_v2 = 0

for resultado in resultados_teste_v2:
    esperadas = converter_em_posicoes(
        resultado["texto"], resultado["esperadas"]
    )
    previstas = converter_em_posicoes(
        resultado["texto"], resultado["previstas"]
    )

    vp_v2 += len(previstas & esperadas)
    fp_v2 += len(previstas - esperadas)
    fn_v2 += len(esperadas - previstas)

    if previstas == esperadas:
        casos_corretos_v2 += 1
    else:
        print(f'REVISAR caso {resultado["caso"]}:')
        print("Esperado:", resultado["esperadas"])
        print("Previsto:", resultado["previstas"])

precisao_v2 = vp_v2 / (vp_v2 + fp_v2) if vp_v2 + fp_v2 else 0
revocacao_v2 = vp_v2 / (vp_v2 + fn_v2) if vp_v2 + fn_v2 else 0
f1_v2 = (
    2 * vp_v2 / (2 * vp_v2 + fp_v2 + fn_v2)
    if 2 * vp_v2 + fp_v2 + fn_v2 else 0
)

print(f"\nCasos sem divergência: {casos_corretos_v2}/{len(resultados_teste_v2)}")
print(f"VP: {vp_v2} | FP: {fp_v2} | FN: {fn_v2}")
print(f"Precisão: {precisao_v2:.2%}")
print(f"Revocação: {revocacao_v2:.2%}")
print(f"F1: {f1_v2:.2%}")

REVISAR caso 6:
Esperado: []
Previsto: [('estagio', 'SERVICO_ACADEMICO')]

Casos sem divergência: 7/8
VP: 9 | FP: 1 | FN: 0
Precisão: 90.00%
Revocação: 100.00%
F1: 94.74%


### Resultado da avaliação adicional da versão 2

Avaliamos a versão 2 em oito novos exemplos sintéticos,
com as anotações propostas antes da execução.

Obtivemos:
- Verdadeiros positivos: 9.
- Falsos positivos: 1.
- Falsos negativos: 0.
- Precisão: 90,00%.
- Revocação: 100,00%.
- F1: 94,74%.

O falso positivo ocorreu em "Eu também estagio em uma empresa".
O filtro não descartou "estagio". A regra que verifica o pronome
"eu" imediatamente anterior não cobre a presença de "também".

Mantivemos as regras inalteradas após essa avaliação.
Os resultados se limitam a um conjunto pequeno e sintético.

As métricas desta avaliação e do piloto inicial foram calculadas
em conjuntos diferentes e não demonstram, isoladamente, uma
melhoria entre versões.

In [31]:
comparacao_versoes = []

for versao in ["v1", "v2"]:
    vp = fp = fn = 0

    for resultado in resultados_teste_v2:
        texto_teste = resultado["texto"]

        esperadas = converter_em_posicoes(
            texto_teste, resultado["esperadas"]
        )

        if versao == "v1":
            documento = nlp_regras(texto_teste)

            previstas = {
                (entidade.start_char, entidade.end_char, entidade.label_)
                for entidade in documento.ents
            }
        else:
            previstas = converter_em_posicoes(
                texto_teste, resultado["previstas"]
            )

        vp += len(previstas & esperadas)
        fp += len(previstas - esperadas)
        fn += len(esperadas - previstas)

    precisao = vp / (vp + fp) if vp + fp else 0
    revocacao = vp / (vp + fn) if vp + fn else 0
    f1 = 2 * vp / (2 * vp + fp + fn) if 2 * vp + fp + fn else 0

    comparacao_versoes.append({
        "versao": versao,
        "VP": vp,
        "FP": fp,
        "FN": fn,
        "precisao": precisao,
        "revocacao": revocacao,
        "f1": f1,
    })

    print(
        f"{versao} | VP={vp} FP={fp} FN={fn} | "
        f"P={precisao:.2%} R={revocacao:.2%} F1={f1:.2%}"
    )

v1 | VP=9 FP=2 FN=0 | P=81.82% R=100.00% F1=90.00%
v2 | VP=9 FP=1 FN=0 | P=90.00% R=100.00% F1=94.74%


In [32]:
import csv
from pathlib import Path

raiz_projeto = Path.cwd()
if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

pasta_v2 = raiz_projeto / "reports" / "ner_contexto_v2"

# Interrompe se a pasta já existir, preservando resultados anteriores.
pasta_v2.mkdir(parents=True, exist_ok=False)

# Uma linha por caso, versão e entidade.
linhas_entidades_v2 = []

for resultado in resultados_teste_v2:
    texto_teste = resultado["texto"]

    documento_v1 = nlp_regras(texto_teste)

    anotacoes = {
        "referencia": converter_em_posicoes(
            texto_teste, resultado["esperadas"]
        ),
        "v1": {
            (entidade.start_char, entidade.end_char, entidade.label_)
            for entidade in documento_v1.ents
        },
        "v2": converter_em_posicoes(
            texto_teste, resultado["previstas"]
        ),
    }

    for origem, entidades_posicionadas in anotacoes.items():
        for inicio, fim, rotulo in sorted(entidades_posicionadas):
            linhas_entidades_v2.append({
                "caso": resultado["caso"],
                "origem": origem,
                "inicio": inicio,
                "fim": fim,
                "texto_entidade": texto_teste[inicio:fim],
                "rotulo": rotulo,
            })

arquivos_v2 = {
    "casos.csv": [
        {"caso": r["caso"], "texto": r["texto"]}
        for r in resultados_teste_v2
    ],
    "entidades.csv": linhas_entidades_v2,
    "comparacao.csv": comparacao_versoes,
}

for nome, linhas in arquivos_v2.items():
    caminho = pasta_v2 / nome

    with caminho.open("x", encoding="utf-8-sig", newline="") as arquivo:
        escritor = csv.DictWriter(
            arquivo,
            fieldnames=list(linhas[0]),
            delimiter=";",
        )
        escritor.writeheader()
        escritor.writerows(linhas)

    with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
        recuperadas = list(csv.DictReader(arquivo, delimiter=";"))

    esperadas = [
        {chave: str(valor) for chave, valor in linha.items()}
        for linha in linhas
    ]
    assert recuperadas == esperadas, f"Falha na conferência de {nome}"

    print(f"{nome}: {len(recuperadas)} registros, conferido")

casos.csv: 8 registros, conferido
entidades.csv: 30 registros, conferido
comparacao.csv: 2 registros, conferido


In [33]:
import spacy

configuracao_v2 = [{
    "versao": "ner_contexto_v2",
    "spacy": spacy.__version__,
    "modelo": nlp.meta["lang"] + "_" + nlp.meta["name"],
    "versao_modelo": nlp.meta["version"],
    "comparacao_texto": "LOWER",
    "termos_filtrados": "matricula|estagio",
    "classe_descartada": "VERB",
    "pronome_anterior_descartado": "eu",
    "alinhamento": "strict",
    "quantidade_casos": len(resultados_teste_v2),
    "tipo_avaliacao": "Exemplos sintéticos",
}]

padroes_v2 = [
    {"label": padrao["label"], "pattern": padrao["pattern"]}
    for padrao in nlp_regras.get_pipe("entity_ruler").patterns
]

assert all(
    isinstance(padrao["pattern"], str)
    for padrao in padroes_v2
), "Esta exportação espera padrões de texto."

for nome, linhas in {
    "configuracao.csv": configuracao_v2,
    "padroes.csv": padroes_v2,
}.items():
    with (pasta_v2 / nome).open(
        "x", encoding="utf-8-sig", newline=""
    ) as arquivo:
        escritor = csv.DictWriter(
            arquivo,
            fieldnames=list(linhas[0]),
            delimiter=";",
        )
        escritor.writeheader()
        escritor.writerows(linhas)

    print(f"{nome}: {len(linhas)} registros salvos")

configuracao.csv: 1 registros salvos
padroes.csv: 12 registros salvos


In [34]:
def ler_csv_v2(nome):
    with (pasta_v2 / nome).open(
        encoding="utf-8-sig", newline=""
    ) as arquivo:
        return list(csv.DictReader(arquivo, delimiter=";"))


config_v2 = ler_csv_v2("configuracao.csv")[0]
casos_salvos_v2 = ler_csv_v2("casos.csv")
entidades_salvas_v2 = ler_csv_v2("entidades.csv")

assert spacy.__version__ == config_v2["spacy"]

modelo_reproducao = spacy.load(config_v2["modelo"])
assert modelo_reproducao.meta["version"] == config_v2["versao_modelo"]

regras_reproducao = spacy.blank(modelo_reproducao.lang)
ruler_reproducao = regras_reproducao.add_pipe(
    "entity_ruler",
    config={"phrase_matcher_attr": config_v2["comparacao_texto"]},
)
ruler_reproducao.add_patterns(ler_csv_v2("padroes.csv"))

termos_filtrados = set(config_v2["termos_filtrados"].split("|"))

for caso_salvo in casos_salvos_v2:
    texto_reproducao = caso_salvo["texto"]
    doc_regras = regras_reproducao(texto_reproducao)
    doc_gramatical = modelo_reproducao(texto_reproducao)

    previsoes_reproduzidas = {"v1": set(), "v2": set()}

    for entidade in doc_regras.ents:
        posicao = (
            entidade.start_char,
            entidade.end_char,
            entidade.label_,
        )
        previsoes_reproduzidas["v1"].add(posicao)

        descartar = False

        if (
            entidade.label_ == "SERVICO_ACADEMICO"
            and entidade.text.lower() in termos_filtrados
        ):
            trecho = doc_gramatical.char_span(
                entidade.start_char,
                entidade.end_char,
                alignment_mode=config_v2["alinhamento"],
            )

            if trecho is not None:
                verbo = any(
                    token.pos_ == config_v2["classe_descartada"]
                    for token in trecho
                )
                pronome_anterior = (
                    trecho.start > 0
                    and doc_gramatical[trecho.start - 1].lower_
                    == config_v2["pronome_anterior_descartado"]
                )
                descartar = verbo or pronome_anterior

        if not descartar:
            previsoes_reproduzidas["v2"].add(posicao)

    for versao in ["v1", "v2"]:
        previsoes_salvas = {
            (int(e["inicio"]), int(e["fim"]), e["rotulo"])
            for e in entidades_salvas_v2
            if e["caso"] == caso_salvo["caso"]
            and e["origem"] == versao
        }

        assert previsoes_reproduzidas[versao] == previsoes_salvas, (
            f'Divergência: caso {caso_salvo["caso"]}, {versao}'
        )

print(
    f"Reprodução conferida: {len(casos_salvos_v2)} casos, "
    "nas versões v1 e v2."
)

Reprodução conferida: 8 casos, nas versões v1 e v2.


## Conclusão

Implementamos uma versão inicial com EntityRuler e uma segunda
versão com filtro gramatical e uma regra de contexto para as
formas ambíguas "matricula" e "estagio".

Nos mesmos oito casos sintéticos de avaliação, obtivemos:

| Versão | VP | FP | FN | Precisão | Revocação | F1 |
|---|---:|---:|---:|---:|---:|---:|
| v1: regras | 9 | 2 | 0 | 81,82% | 100% | 90,00% |
| v2: regras e contexto | 9 | 1 | 0 | 90,00% | 100% | 94,74% |

O filtro reduziu um falso positivo e preservou os acertos.
Permaneceu o reconhecimento indevido de "estagio" em
"Eu também estagio em uma empresa".

Os resultados se limitam a exemplos sintéticos e a um vocabulário
restrito. Não demonstram o desempenho em perguntas reais.
A análise gramatical pode errar e as regras de contexto têm
cobertura limitada.

Salvamos os casos, entidades, métricas comparativas, padrões e
configuração em reports/ner_contexto_v2. Reconstruímos os
reconhecedores e reproduzimos as previsões das duas versões
nos oito casos.